# SASRec Stage 3 Baseline Multi-Task BPI2012 Colab Train 02

This notebook is the Stage 3 follow-up version.

Main changes from `01`:
- reuse all available single-task baseline seeds (`42`, `2024`, `7`)
- compare multi-task runs with the same 3 seeds
- inspect mean/std instead of only `s42`
- show task metrics with the final metric names used in later experiments

Main comparison metric:
- `full ranking + NDCG@10`


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.11.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
MULTITASK_BASELINE_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('MULTITASK_BASELINE_OUTPUT_DIR:', MULTITASK_BASELINE_OUTPUT_DIR)
print('NOTEBOOK_DIR:', NOTEBOOK_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
MULTITASK_BASELINE_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10
NOTEBOOK_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/notebooks


In [6]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$MULTITASK_BASELINE_OUTPUT_DIR"


In [7]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [8]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [9]:
!pip install -r requirements_colab.txt


In [10]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [11]:
%cd /content/time-aware-behavior-prediction
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Verify Stage 3 processed file

Stage 3 next-time prediction uses the processed time-feature CSV.
This check confirms that `delta_next_seconds` already exists.


In [12]:
import pandas as pd

time_features_path = 'data/processed/bpi2012_complete_only/events_encoded_time_features.csv'
df = pd.read_csv(time_features_path)
required_cols = [
    'delta_prev_seconds',
    'delta_start_seconds',
    'delta_next_seconds',
]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(f'Missing required Stage 3 columns: {missing}')

print('Stage 3 processed file is ready.')
print(df.columns.tolist())
df[['user_id', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds']].head()


Stage 3 processed file is ready.
['case_id', 'activity', 'lifecycle', 'timestamp', 'event_idx', 'delta_prev_seconds', 'delta_start_seconds', 'delta_next_seconds', 'user_id', 'item_id']


,user_id,event_idx,delta_prev_seconds,delta_start_seconds,delta_next_seconds
0,1,0,0.000,0.000,0.334
1,1,1,0.334,0.334,53.026
2,1,2,53.026,53.360,39785.402
3,1,3,39785.402,39838.762,145.935
4,1,4,145.935,39984.697,-0.000


## Experiment design

Stage 3 baseline multi-task runs:

- backbone 1: `anchor_ml20`
- backbone 2: `refine_ml50_do035`
- multi-task outputs: `next activity + next time`
- time target: `delta_next_seconds`
- time target transform: `log1p`
- time loss: `huber`
- time loss weight: `1.0`
- best epoch criterion: `full_valid_ndcg@10`
- final comparison should use all 3 seeds when available: `42`, `2024`, `7`


## Check prerequisite baseline runs (all 3 seeds)


In [13]:
from pathlib import Path

baseline_required_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

output_dir = Path(BASELINE_NDCG10_OUTPUT_DIR)
print('=' * 80)
print('Baseline NDCG@10 prerequisite runs')
for run_name in baseline_required_runs:
    run_dir = output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10 prerequisite runs
anchor_ml20_s42 EXISTS
anchor_ml20_s2024 EXISTS
anchor_ml20_s7 EXISTS
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS


## Check planned multi-task runs


In [14]:
planned_multitask_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]

output_dir = Path(MULTITASK_BASELINE_OUTPUT_DIR)
print('=' * 80)
print('Stage 3 baseline multi-task runs')
for run_name in planned_multitask_runs:
    run_dir = output_dir / run_name
    print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Stage 3 baseline multi-task runs
multitask_anchor_ml20_s42 EXISTS
multitask_anchor_ml20_s2024 EXISTS
multitask_anchor_ml20_s7 EXISTS
multitask_refine_ml50_do035_s42 EXISTS
multitask_refine_ml50_do035_s2024 EXISTS
multitask_refine_ml50_do035_s7 EXISTS


## Train multi-task runs


In [15]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10/multitask_anchor_ml20_s42
epoch=1, loss=2.5763
epoch=2, loss=1.5438
epoch=3, loss=1.4283
epoch=4, loss=1.3598
epoch=5, loss=1.3169
valid [task], Top5Acc: 0.4004, Top10Acc: 0.7708, Acc: 0.0551, MacroF1: 0.0988, TimeMAE: 70257.4629, TimeRMSE: 275813.0764, TimeMedAE: 878.5276
valid [full], NDCG@5: 0.6264, HR@5: 0.7599, NDCG@10: 0.6882, HR@10: 0.9589, MRR: 0.6119
valid [sampled], NDCG@5: 0.5231, HR@5: 0.5234, NDCG@10: 0.5258, HR@10: 0.5317, MRR: 0.5389
test [task], Top5Acc: 0.0465, Top10Acc: 0.5384, Acc: 0.0141, MacroF1: 0.0150, TimeMAE: 11838.0120, Ti

In [16]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10/multitask_anchor_ml20_s2024
epoch=1, loss=2.8066
epoch=2, loss=1.5922
epoch=3, loss=1.4238
epoch=4, loss=1.3530
epoch=5, loss=1.3070
valid [task], Top5Acc: 0.4705, Top10Acc: 0.7222, Acc: 0.0877, MacroF1: 0.1529, TimeMAE: 76237.8869, TimeRMSE: 291919.8238, TimeMedAE: 1167.8160
valid [full], NDCG@5: 0.6216, HR@5: 0.7224, NDCG@10: 0.6944, HR@10: 0.9546, MRR: 0.6220
valid [sampled], NDCG@5: 0.5316, HR@5: 0.5323, NDCG@10: 0.5372, HR@10: 0.5497, MRR: 0.5476
test [task], Top5Acc: 0.0602, Top10Acc: 0.4795, Acc: 0.0359, MacroF1: 0.0325, TimeMAE: 12705.2599,

In [17]:
!python src/train_sasrec.py \
  --run_name multitask_anchor_ml20_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10/multitask_anchor_ml20_s7
epoch=1, loss=2.5159
epoch=2, loss=1.5425
epoch=3, loss=1.4187
epoch=4, loss=1.3537
epoch=5, loss=1.3111
valid [task], Top5Acc: 0.3653, Top10Acc: 0.7063, Acc: 0.0763, MacroF1: 0.1307, TimeMAE: 71307.8360, TimeRMSE: 288739.1460, TimeMedAE: 716.9146
valid [full], NDCG@5: 0.6091, HR@5: 0.7073, NDCG@10: 0.6951, HR@10: 0.9818, MRR: 0.6128
valid [sampled], NDCG@5: 0.5145, HR@5: 0.5171, NDCG@10: 0.5248, HR@10: 0.5495, MRR: 0.5310
test [task], Top5Acc: 0.3190, Top10Acc: 0.4255, Acc: 0.0480, MacroF1: 0.0334, TimeMAE: 12699.1679, Tim

In [18]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10/multitask_refine_ml50_do035_s42
epoch=1, loss=2.8505
epoch=2, loss=1.7178
epoch=3, loss=1.5850
epoch=4, loss=1.4952
epoch=5, loss=1.4479
valid [task], Top5Acc: 0.3369, Top10Acc: 0.7661, Acc: 0.0630, MacroF1: 0.0964, TimeMAE: 68718.9606, TimeRMSE: 276894.8275, TimeMedAE: 685.3123
valid [full], NDCG@5: 0.5947, HR@5: 0.7072, NDCG@10: 0.6826, HR@10: 0.9763, MRR: 0.5979
valid [sampled], NDCG@5: 0.5073, HR@5: 0.5086, NDCG@10: 0.5090, HR@10: 0.5139, MRR: 0.5224
test [task], Top5Acc: 0.0402, Top10Acc: 0.4269, Acc: 0.0250, MacroF1: 0.0223, TimeMAE: 11325.08

In [19]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10/multitask_refine_ml50_do035_s2024
epoch=1, loss=3.1766
epoch=2, loss=1.7923
epoch=3, loss=1.5831
epoch=4, loss=1.4831
epoch=5, loss=1.4271
valid [task], Top5Acc: 0.3530, Top10Acc: 0.6978, Acc: 0.0731, MacroF1: 0.1150, TimeMAE: 72415.8090, TimeRMSE: 292456.3123, TimeMedAE: 1158.1624
valid [full], NDCG@5: 0.6044, HR@5: 0.7012, NDCG@10: 0.6672, HR@10: 0.9041, MRR: 0.6062
valid [sampled], NDCG@5: 0.5189, HR@5: 0.5195, NDCG@10: 0.5233, HR@10: 0.5332, MRR: 0.5340
test [task], Top5Acc: 0.2852, Top10Acc: 0.5307, Acc: 0.0203, MacroF1: 0.0191, TimeMAE: 12778

In [20]:
!python src/train_sasrec.py \
  --run_name multitask_refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --enable_time_prediction \
  --time_prediction_target delta_next_seconds \
  --time_target_transform log1p \
  --time_loss_type huber \
  --time_loss_weight 1.0 \
  --output_dir "$MULTITASK_BASELINE_OUTPUT_DIR" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --time_features_path data/processed/bpi2012_complete_only/events_encoded_time_features.csv \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_stage3_baseline_multitask_ndcg10/multitask_refine_ml50_do035_s7
epoch=1, loss=2.8340
epoch=2, loss=1.7338
epoch=3, loss=1.5919
epoch=4, loss=1.5063
epoch=5, loss=1.4528
valid [task], Top5Acc: 0.3922, Top10Acc: 0.7020, Acc: 0.0631, MacroF1: 0.1005, TimeMAE: 72185.0271, TimeRMSE: 289690.4072, TimeMedAE: 1009.0027
valid [full], NDCG@5: 0.4991, HR@5: 0.6484, NDCG@10: 0.5953, HR@10: 0.9491, MRR: 0.4925
valid [sampled], NDCG@5: 0.3377, HR@5: 0.3638, NDCG@10: 0.3751, HR@10: 0.4805, MRR: 0.3601
test [task], Top5Acc: 0.3540, Top10Acc: 0.6121, Acc: 0.0312, MacroF1: 0.0253, TimeMAE: 14388.22

In [21]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'enable_time_prediction': config.get('enable_time_prediction', False),
            'time_prediction_target': config.get('time_prediction_target'),
            'time_loss_weight': config.get('time_loss_weight'),
            'time_target_transform': config.get('time_target_transform'),
            'time_modeling_mode': config.get('time_modeling_mode'),
        }
        for group_name in ['best_valid', 'best_test_at_best_valid', 'last_valid', 'last_test']:
            group = summary.get(group_name) or {}
            for mode, metrics in group.items():
                for key, value in metrics.items():
                    row[f'{group_name}_{mode}_{key}'] = value
        rows.append(row)
    return pd.DataFrame(rows)


In [22]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1600)
pd.set_option('display.max_colwidth', None)


## Comparison summary


In [25]:
baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
multitask_runs = [
    'multitask_anchor_ml20_s42',
    'multitask_anchor_ml20_s2024',
    'multitask_anchor_ml20_s7',
    'multitask_refine_ml50_do035_s42',
    'multitask_refine_ml50_do035_s2024',
    'multitask_refine_ml50_do035_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
multitask_df = rebuild_df(MULTITASK_BASELINE_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['variant'] = baseline_subset['run_name'].map(
    lambda x: 'anchor_single_task' if x.startswith('anchor_ml20') else 'refine_single_task'
)

multitask_subset = multitask_df[multitask_df['run_name'].isin(multitask_runs)].copy()
multitask_subset['variant'] = multitask_subset['run_name'].map(
    lambda x: 'anchor_multi_task' if 'anchor_ml20' in x else 'refine_multi_task'
)

df_compare = pd.concat([baseline_subset, multitask_subset], ignore_index=True)
df_compare = df_compare.sort_values(['variant', 'seed', 'run_name']).reset_index(drop=True)

display_cols = [
    'run_name', 'seed', 'variant', 'maxlen', 'dropout_rate', 'selection_metric',
    'best_epoch',

    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_mrr',

    'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10',
    'best_test_at_best_valid_full_mrr',

    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_mrr',

    'best_test_at_best_valid_sampled_ndcg@5', 'best_test_at_best_valid_sampled_hr@5',
    'best_test_at_best_valid_sampled_ndcg@10', 'best_test_at_best_valid_sampled_hr@10',
    'best_test_at_best_valid_sampled_mrr',

    'best_valid_task_accuracy',
    'best_valid_task_macro_f1',
    'best_valid_task_top5_accuracy',
    'best_valid_task_top10_accuracy',
    'best_valid_task_time_mae',
    'best_valid_task_time_rmse',
    'best_valid_task_time_median_ae',

    'best_test_at_best_valid_task_accuracy',
    'best_test_at_best_valid_task_macro_f1',
    'best_test_at_best_valid_task_top5_accuracy',
    'best_test_at_best_valid_task_top10_accuracy',
    'best_test_at_best_valid_task_time_mae',
    'best_test_at_best_valid_task_time_rmse',
    'best_test_at_best_valid_task_time_median_ae',
]

existing_display_cols = [c for c in display_cols if c in df_compare.columns]
df_compare[existing_display_cols]



,run_name,seed,variant,maxlen,dropout_rate,selection_metric,best_epoch,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_mrr,best_test_at_best_valid_full_ndcg@5,best_test_at_best_valid_full_hr@5,best_test_at_best_valid_full_ndcg@10,best_test_at_best_valid_full_hr@10,best_test_at_best_valid_full_mrr,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_mrr,best_test_at_best_valid_sampled_ndcg@5,best_test_at_best_valid_sampled_hr@5,best_test_at_best_valid_sampled_ndcg@10,best_test_at_best_valid_sampled_hr@10,best_test_at_best_valid_sampled_mrr,best_valid_task_accuracy,best_valid_task_macro_f1,best_valid_task_top5_accuracy,best_valid_task_top10_accuracy,best_valid_task_time_mae,best_valid_task_time_rmse,best_valid_task_time_median_ae,best_test_at_best_valid_task_accuracy,best_test_at_best_valid_task_macro_f1,best_test_at_best_valid_task_top5_accuracy,best_test_at_best_valid_task_top10_accuracy,best_test_at_best_valid_task_time_mae,best_test_at_best_valid_task_time_rmse,best_test_at_best_valid_task_time_median_ae
0,multitask_anchor_ml20_s7,7,anchor_multi_task,20,0.20,full_valid_ndcg@10,45,0.718860,0.876119,0.739533,0.944505,0.678634,0.730814,0.939099,0.749556,0.999865,0.665654,0.557397,0.561872,0.572765,0.610312,0.578015,0.252553,0.296928,0.314915,0.491271,0.287664,0.062008,0.092260,0.402035,0.608412,71386.674438,287652.313094,781.026734,0.023143,0.021551,0.465692,0.560157,11294.567416,66736.008111,6.871000
1,multitask_anchor_ml20_s42,42,anchor_multi_task,20,0.20,full_valid_ndcg@10,30,0.706125,0.856119,0.742361,0.976065,0.671685,0.785379,0.929319,0.809234,1.000000,0.745731,0.538513,0.545639,0.554949,0.597431,0.559410,0.323805,0.342939,0.366958,0.477912,0.363845,0.053685,0.052223,0.408384,0.692765,74664.950418,282278.030256,6863.672687,0.014680,0.011972,0.478864,0.668207,14168.895027,78819.227077,129.408642
2,multitask_anchor_ml20_s2024,2024,anchor_multi_task,20,0.20,full_valid_ndcg@10,40,0.671279,0.798867,0.718543,0.945636,0.653637,0.828095,0.992412,0.830744,1.000000,0.772635,0.557500,0.559962,0.565643,0.585593,0.574222,0.289004,0.306098,0.332764,0.444580,0.334374,0.070282,0.069061,0.402536,0.648051,76033.382424,296497.595071,1482.530225,0.116531,0.051538,0.476152,0.664770,12305.288892,71731.711803,57.030022
3,anchor_ml20_s7,7,anchor_single_task,20,0.20,full_valid_ndcg@10,30,0.729167,0.883942,0.753294,0.961857,0.691225,0.903861,0.971918,0.913637,1.000000,0.885501,0.580044,0.594609,0.601240,0.661113,0.597937,0.420861,0.484209,0.480200,0.667438,0.443705,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,anchor_ml20_s42,42,anchor_single_task,20,0.20,full_valid_ndcg@10,50,0.708784,0.848505,0.750367,0.977546,0.681799,0.913814,0.995781,0.915296,1.000000,0.886842,0.556357,0.567822,0.583692,0.654340,0.578433,0.406641,0.473262,0.464891,0.652470,0.430220,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,anchor_ml20_s2024,2024,anchor_single_task,20,0.20,full_valid_ndcg@10,25,0.700301,0.850202,0.736821,0.967429,0.668534,0.819421,0.937829,0.840611,0.999865,0.789072,0.557801,0.563197,0.574314,0.616064,0.578608,0.261120,0.326822,0.323103,0.518524,0.292464,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,multitask_refine_ml50_do035_s7,7,refine_multi_task,50,0.35,full_valid_ndcg@10,15,0.654646,0.714809,0.731297,0.956020,0.669411,0.654655,0.851278,0.703583,1.000000,0.608590,0.597831,0.598208,0.601645,0.610289,0.610293,0.169432,0.172104,0.196568,0.259516,0.217373,0.064612,0.094257,0.449165,0.769105,72160.363915,279006.648888,773.954303,0.022567,0.020475,0.330479,0.620582,12943.153281,79617.768250,17.894858
7,multitask_refine_ml50_do035_s42,42,refine_multi_task,50,0.35,full_valid_ndcg@10,20,0.664381,0.749320,0.732935,0.969141,0.667017,0.652864,0.939369,0.672353,1.000000,0.563962,0.600481,0.600870,0.602203,0.606444,0.611856,0.068749,0.072405,0.101780,0.178779,0.123676,0.066340,0.105943,0.448206,0.703371,749

In [26]:
summary_metric_cols = [
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_mrr',

    'best_test_at_best_valid_full_ndcg@5', 'best_test_at_best_valid_full_hr@5',
    'best_test_at_best_valid_full_ndcg@10', 'best_test_at_best_valid_full_hr@10',
    'best_test_at_best_valid_full_mrr',

    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_mrr',

    'best_test_at_best_valid_sampled_ndcg@5', 'best_test_at_best_valid_sampled_hr@5',
    'best_test_at_best_valid_sampled_ndcg@10', 'best_test_at_best_valid_sampled_hr@10',
    'best_test_at_best_valid_sampled_mrr',

    'best_valid_task_accuracy',
    'best_valid_task_macro_f1',
    'best_valid_task_top5_accuracy',
    'best_valid_task_top10_accuracy',
    'best_valid_task_time_mae',
    'best_valid_task_time_rmse',
    'best_valid_task_time_median_ae',

    'best_test_at_best_valid_task_accuracy',
    'best_test_at_best_valid_task_macro_f1',
    'best_test_at_best_valid_task_top5_accuracy',
    'best_test_at_best_valid_task_top10_accuracy',
    'best_test_at_best_valid_task_time_mae',
    'best_test_at_best_valid_task_time_rmse',
    'best_test_at_best_valid_task_time_median_ae',
]

summary_metric_cols = [c for c in summary_metric_cols if c in df_compare.columns]
summary_compare = df_compare.groupby('variant')[summary_metric_cols].agg(['mean', 'std'])
summary_compare



best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_mrr           best_test_at_best_valid_full_ndcg@5           best_test_at_best_valid_full_hr@5           best_test_at_best_valid_full_ndcg@10           best_test_at_best_valid_full_hr@10           best_test_at_best_valid_full_mrr           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_mrr           best_test_at_best_valid_sampled_ndcg@5           best_test_at_best_valid_sampled_hr@5           best_test_at_best_valid_sampled_ndcg@10           best_test_at_best_valid_sampled_hr@10           best_test_at_best_valid_sampled_mrr           best_valid_task_accuracy           best_valid_task_macro_f1           best_valid_task_top5_accuracy          best_valid_task_top10_accuracy           best_valid_task_time_mae              best_valid_task_time_rmse              best_valid_task_time_median_ae              best_test_at_best_valid_task_accuracy           best_test_at_best_valid_task_macro_f1           best_test_at_best_valid_task_top5_accuracy           best_test_at_best_valid_task_top10_accuracy           best_test_at_best_valid_task_time_mae              best_test_at_best_valid_task_time_rmse              best_test_at_best_valid_task_time_median_ae           
                                     mean       std                 mean       std                    mean       std                  mean       std                mean       std                                mean       std                              mean       std                                 mean       std                               mean       std                             mean       std                      mean       std                    mean       std                       mean       std                     mean       std                   mean       std                                   mean       std                                 mean       std                                    mean       std                                  mean       std                                mean       std                     mean       std                     mean       std                          mean      std                           mean       std                     mean          std                      mean          std                           mean          std                                  mean       std                                  mean       std                                       mean       std                                        mean       std                                  mean          std                                   mean          std                                        mean        std
variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

Interpretation guide:

- compare `anchor_single_task` vs `anchor_multi_task`
- compare `refine_single_task` vs `refine_multi_task`
- use `best_test_at_best_valid_full_ndcg@10` as the main Stage 3 comparison metric
- use mean/std across `42`, `2024`, `7` for final interpretation
- task metrics are shown directly as `accuracy`, `macro_f1`, `top5_accuracy`, `top10_accuracy`, `time_mae`, `time_rmse`
